In [ ]:
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import numpy as np
from tqdm.notebook import tqdm # Import tqdm for progress bar
import gc # Import garbage collector

# --- 1. Load Pre-split Test Data ---

# Load the test datasets provided by the user
test_ratings_small = pd.read_csv('/content/test_small.csv')
test_ratings_big = pd.read_csv('/content/test_big.csv')

# Ensure tmdbId and userId are integers for consistency and mapping
test_ratings_small['tmdbId'] = test_ratings_small['tmdbId'].astype(int)
test_ratings_small['userId'] = test_ratings_small['userId'].astype(int)
test_ratings_big['tmdbId'] = test_ratings_big['tmdbId'].astype(int)
test_ratings_big['userId'] = test_ratings_big['userId'].astype(int)

print(f"Loaded test_ratings_small with {len(test_ratings_small):,} ratings.")
print(f"Loaded test_ratings_big with {len(test_ratings_big):,} ratings.")

# Minimum number of items a test user must have in their 'to_predict' set
# to be included in the evaluation, as requested by the user.
MIN_PREDICT_ITEMS_FOR_TEST_USER = 3 # User must have >=3 relevant items in their individual 20% test set

# Minimum number of ratings a user must have in the global test set to be considered for evaluation.
MIN_RATINGS_IN_GLOBAL_TEST = 3 # User must have >=3 ratings in test_small/big to be an eligible user

# Number of most similar users to consider for recommendations
N_SIMILAR_USERS = 100

# Add the new constant for minimum training ratings for evaluation users
MIN_TRAIN_RATINGS_FOR_EVAL_USER = 10 # User must have at least 10 ratings in their individual 80% train set to be evaluated

print(f"Minimum prediction items per test user for evaluation: {MIN_PREDICT_ITEMS_FOR_TEST_USER}")
print(f"Minimum ratings in global test set for user eligibility: {MIN_RATINGS_IN_GLOBAL_TEST}")
print(f"Considering {N_SIMILAR_USERS} most similar users for recommendations.")
print(f"Minimum training ratings for a user to be evaluated: {MIN_TRAIN_RATINGS_FOR_EVAL_USER}")

# --- 2. Prepare Data for Recommendation System (UBCF Model Training) ---

# For UBCF, the model (i.e., the user-item matrix and NN model) is trained on the ENTIRE 'ratings' dataset.
# The per-user 80/20 split will be handled during the evaluation loop to define 'known_movie_ids'.
full_ubcf_train_data = ratings.copy() # Use the entire ratings DataFrame for building the UBCF matrix

# Map original IDs to continuous indices using the full ratings data
all_unique_users_ubcf = full_ubcf_train_data['userId'].unique()
all_unique_tmdb_ids_ubcf = full_ubcf_train_data['tmdbId'].unique()

user_id_map_ubcf = {uid: i for i, uid in enumerate(all_unique_users_ubcf)}
tmdb_id_map_ubcf = {mid: i for i, mid in enumerate(all_unique_tmdb_ids_ubcf)}

n_users_total_ubcf = len(all_unique_users_ubcf)
n_items_total_ubcf = len(all_unique_tmdb_ids_ubcf)

# Apply mapping to the full_ubcf_train_data DataFrame to build the matrix for training UBCF
full_ubcf_train_data_mapped = full_ubcf_train_data.copy()
full_ubcf_train_data_mapped['user_idx'] = full_ubcf_train_data_mapped['userId'].map(user_id_map_ubcf)
full_ubcf_train_data_mapped['item_idx'] = full_ubcf_train_data_mapped['tmdbId'].map(tmdb_id_map_ubcf)

# Drop rows where user_idx or item_idx could not be mapped (should not happen if using all ratings for map creation)
full_ubcf_train_data_mapped.dropna(subset=['user_idx', 'item_idx'], inplace=True)
full_ubcf_train_data_mapped['user_idx'] = full_ubcf_train_data_mapped['user_idx'].astype(int)
full_ubcf_train_data_mapped['item_idx'] = full_ubcf_train_data_mapped['item_idx'].astype(int)

# Create a Compressed Sparse Row (CSR) matrix for memory efficiency
train_user_item_matrix_ubcf = csr_matrix((full_ubcf_train_data_mapped['rating'],
                                          (full_ubcf_train_data_mapped['user_idx'],
                                           full_ubcf_train_data_mapped['item_idx'])),
                                         shape=(n_users_total_ubcf, n_items_total_ubcf))
print("UBCF user-item matrix created using sparse format from the entire ratings data.")

# Clear memory from temporary mapped ratings DataFrame
del full_ubcf_train_data_mapped
del full_ubcf_train_data
gc.collect()

# --- 3. User-Based Collaborative Filtering (NearestNeighbors for scalability) ---

# Initialize NearestNeighbors model to find similar users.
print(f"Fitting NearestNeighbors model with {N_SIMILAR_USERS} similar users using cosine metric...")
nn_model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=N_SIMILAR_USERS + 1, n_jobs=-1)
nn_model.fit(train_user_item_matrix_ubcf)
print("NearestNeighbors model fitted.")

# Pre-calculate user average ratings from the GLOBAL training data for efficient lookup during prediction
# This should also be based on the entire 'ratings' dataframe for consistency if the matrix is built on it.
user_avg_ratings_eval_map_ubcf = ratings.groupby('userId')['rating'].mean().to_dict()

# Create a reverse user ID map for looking up original user IDs from user_idx (global)
reverse_user_id_map_ubcf = {v: k for k, v in user_id_map_ubcf.items()}

# Calculate item popularity (average rating) from the *full* ratings dataset for serendipity
movie_popularity_ubcf = ratings.groupby('tmdbId')['rating'].mean()
# Normalize popularity to a range between 0 and 1 by dividing by max possible rating (5.0)
normalized_movie_popularity_ubcf = (movie_popularity_ubcf / 5.0).to_dict()
print("Normalized movie popularity calculated for serendipity metric using full ratings data.")

def get_user_recommendations_optimized_enhanced(
    target_user_id, train_matrix, nn_model, user_id_map, tmdb_id_map,
    n_recommendations=10, user_watched_movie_ids=None, N_SIMILAR_USERS_PARAM=100,
    user_avg_ratings_map_for_prediction=None, reverse_user_id_map=None,
    normalized_movie_popularity_map=None
):
    """
    Generates recommendations for a target user using NearestNeighbors for finding similar users
    on raw ratings, with neighbor mean-centering during prediction and popularity weighting.

    Args:
        target_user_id (int): Original ID of the user for whom to generate recommendations.
        train_matrix (csr_matrix): Sparse matrix of RAW ratings (built from full dataset).
        nn_model (NearestNeighbors): Fitted NearestNeighbors model.
        user_id_map (dict): Mapping from original userId to internal user_idx.
        tmdb_id_map (dict): Mapping from original tmdbId to internal item_idx.
        n_recommendations (int): The number of recommendations to return.
        user_watched_movie_ids (set): Set of tmdbIds the target user has already rated (from their individual train set).
        N_SIMILAR_USERS_PARAM (int): Number of most similar users to consider.
        user_avg_ratings_map_for_prediction (dict): Map of userId to their average rating for prediction baseline.
        reverse_user_id_map (dict): Reverse map from internal user_idx to original userId.
        normalized_movie_popularity_map (dict): Map of tmdbId to normalized popularity score.

    Returns:
        list: A list of recommended movie tmdbIds.
    """
    if user_watched_movie_ids is None or user_avg_ratings_map_for_prediction is None or reverse_user_id_map is None or normalized_movie_popularity_map is None:
        raise ValueError("All required maps (user_watched_movie_ids, user_avg_ratings_map_for_prediction, reverse_user_id_map, normalized_movie_popularity_map) must be provided.")

    target_user_idx = user_id_map.get(target_user_id)
    if target_user_idx is None or target_user_idx >= train_matrix.shape[0]:
        return []

    # Use NearestNeighbors to find similar users
    distances, indices = nn_model.kneighbors(train_matrix[target_user_idx], n_neighbors=N_SIMILAR_USERS_PARAM + 1)

    # Convert distances to similarities (1 - distance for cosine) and filter out the target user itself
    similar_user_indices = indices[0][1:]
    similar_user_distances = distances[0][1:]
    similar_user_scores = 1 - similar_user_distances # Convert cosine distance to similarity

    # Filter out users with non-positive similarity (can happen if distance >= 1)
    valid_similar_users = [(sim_idx, score) for sim_idx, score in zip(similar_user_indices, similar_user_scores) if score > 0]

    if not valid_similar_users:
        return []

    item_scores = {} # This will store the aggregated prediction scores
    reverse_tmdb_id_map = {v: k for k, v in tmdb_id_map.items()}

    for sim_user_idx, sim_score in valid_similar_users:
        sim_user_sparse_row = train_matrix[sim_user_idx] # Get sparse row with raw ratings

        sim_user_original_id = reverse_user_id_map.get(sim_user_idx)
        if sim_user_original_id is None:
            continue

        # Get similar user's average rating from the provided map (which should be the GLOBAL average ratings)
        sim_user_avg_rating = user_avg_ratings_map_for_prediction.get(sim_user_original_id, 0.0)

        for item_idx, raw_rating in zip(sim_user_sparse_row.indices, sim_user_sparse_row.data):
            tmdb_id = reverse_tmdb_id_map.get(item_idx)

            # Only consider items not yet rated by the target user AND are valid tmdbIds from our map
            if tmdb_id is not None and tmdb_id not in user_watched_movie_ids:
                # Weighted sum of centered ratings from neighbors, then boosted by popularity
                centered_neighbor_rating = raw_rating - sim_user_avg_rating
                popularity_score = normalized_movie_popularity_map.get(tmdb_id, 0.0) # Get normalized popularity, default to 0 if not found
                weighting_factor = 1 + popularity_score

                item_scores[item_idx] = item_scores.get(item_idx, 0) + sim_score * centered_neighbor_rating * weighting_factor

    # Sort items by their aggregated score and take the top N recommendations
    recommended_item_indices = sorted(item_scores, key=item_scores.get, reverse=True)[:n_recommendations]

    # Map item indices back to original tmdbIds for evaluation
    recommendations_tmdb_ids = [reverse_tmdb_id_map[idx] for idx in recommended_item_indices if idx in reverse_tmdb_id_map]

    return recommendations_tmdb_ids

print("Recommendation function (NearestNeighbors for similar users, neighbor mean-centering, popularity weighting) defined.")

# --- 4. Evaluation Metrics Functions ---
def precision_at_k(recommended_items, actual_items, k):
    """Calculates Precision@k."""
    recommended_set = set(recommended_items[:k])
    actual_set = set(actual_items)
    if not recommended_set:
        return 0.0
    return len(recommended_set.intersection(actual_set)) / k

def recall_at_k(recommended_items, actual_items, k):
    """Calculates Recall@k."""
    recommended_set = set(recommended_items[:k])
    actual_set = set(actual_items)
    if not actual_set: # Cannot recall anything if there's nothing to recall
        return 0.0
    return len(recommended_set.intersection(actual_set)) / len(actual_set)

def calculate_dcg(relevance_scores, k):
    dcg = 0.0
    for i in range(min(k, len(relevance_scores))):
        dcg += relevance_scores[i] / np.log2(i + 2)
    return dcg

def ndcg_at_k(recommended_items, actual_items, k):
    """Calculates Normalized Discounted Cumulative Gain (NDCG@k)."""
    # Create binary relevance scores for recommended items
    relevance_scores = [1 if item in actual_items else 0 for item in recommended_items]

    # Calculate DCG for the recommended list
    dcg = calculate_dcg(relevance_scores, k)

    # Calculate IDCG (Ideal DCG) for the ground truth relevant items
    # First, create ideal relevance scores (all relevant items first)
    ideal_relevance_scores = [1] * len(actual_items)
    idcg = calculate_dcg(ideal_relevance_scores, k)

    if idcg == 0:
        return 0.0  # Avoid division by zero if there are no relevant items
    return dcg / idcg

def calculate_serendipity(recommended_items, actual_relevant_items, normalized_movie_popularity_map, k):
    """
    Calculates serendipity@k for a given user.
    Serendipity = sum(Relevance_item * (1 - Normalized_Popularity_item)) / k
    """
    serendipity_score = 0.0
    recommended_set = set(recommended_items[:k])
    actual_relevant_set = set(actual_relevant_items)

    for item in recommended_set:
        if item in actual_relevant_set: # Item is relevant
            popularity = normalized_movie_popularity_map.get(item, 0.0) # Get normalized popularity, default to 0 if not found
            serendipity_score += (1 - popularity)
    return serendipity_score / k if k > 0 else 0.0


print("Evaluation metrics functions (Precision, Recall, NDCG, Serendipity) defined.")
gc.collect() # Final garbage collection after setup

Loaded test_ratings_small with 25,930 ratings.
Loaded test_ratings_big with 518,593 ratings.
Minimum prediction items per test user for evaluation: 3
Minimum ratings in global test set for user eligibility: 3
Considering 100 most similar users for recommendations.
Minimum training ratings for a user to be evaluated: 10
UBCF user-item matrix created using sparse format from the entire ratings data.
Fitting NearestNeighbors model with 100 similar users using cosine metric...
NearestNeighbors model fitted.
Normalized movie popularity calculated for serendipity metric using full ratings data.
Recommendation function (NearestNeighbors for similar users, neighbor mean-centering, popularity weighting) defined.
Evaluation metrics functions (Precision, Recall, NDCG, Serendipity) defined.


0

In [ ]:
print('Executing evaluation for test_small.csv (UBCF)')
k_values = [5, 10, 15]
all_precision_small_ubcf = {k: [] for k in k_values}
all_recall_small_ubcf = {k: [] for k in k_values}
all_ndcg_small_ubcf = {k: [] for k in k_values}
all_serendipity_small_ubcf = {k: [] for k in k_values}

# 1. Identify users for evaluation from the GLOBAL test set
# These are users who have at least MIN_RATINGS_IN_GLOBAL_TEST ratings in the 'test_small.csv' file
test_user_counts_small_ubcf = test_ratings_small['userId'].value_counts()
eligible_users_small_ubcf = test_user_counts_small_ubcf[test_user_counts_small_ubcf >= MIN_RATINGS_IN_GLOBAL_TEST].index.tolist()

print(f"\nStarting evaluation for UBCF on test_small.csv (per-user 80/20 split), considering {len(eligible_users_small_ubcf):,} eligible test users...")

processed_users_count_small_ubcf = 0
for user_id in tqdm(eligible_users_small_ubcf, desc="Evaluating UBCF on test_small.csv (per-user split)"):
    # Get ALL ratings for this specific user from the FULL 'ratings' dataset
    user_all_history = ratings[ratings['userId'] == user_id].sort_values(by='timestamp').reset_index(drop=True)

    # Perform 80/20 chronological split for THIS USER'S ENTIRE HISTORY
    user_split_point = int(len(user_all_history) * 0.8)
    user_train_individual = user_all_history.iloc[:user_split_point]
    user_test_individual = user_all_history.iloc[user_split_point:]

    # Ensure individual train set is not empty and has enough ratings for model training
    if user_train_individual.empty or len(user_train_individual) < MIN_TRAIN_RATINGS_FOR_EVAL_USER:
        continue

    # Known movies for this user (from their individual 80% train set)
    known_movie_ids = set(user_train_individual['tmdbId'])
    user_avg_rating = user_train_individual['rating'].mean()

    # Define 'relevant' items for ground truth from the INDIVIDUAL user's 20% test set
    # Rating >= 4 AND Rating >= user's average rating from their train set
    ground_truth_relevant_items = user_test_individual[
        (user_test_individual['rating'] >= 4) &
        (user_test_individual['rating'] >= user_avg_rating)
    ]['tmdbId'].tolist()

    # Only evaluate user if their individual test set has at least MIN_PREDICT_ITEMS_FOR_TEST_USER relevant items
    if not ground_truth_relevant_items or len(ground_truth_relevant_items) < MIN_PREDICT_ITEMS_FOR_TEST_USER:
        continue

    processed_users_count_small_ubcf += 1

    max_k = max(k_values)
    recommended_items = get_user_recommendations_optimized_enhanced(
        user_id,
        train_user_item_matrix_ubcf, # Use the global UBCF matrix
        nn_model,
        user_id_map_ubcf,
        tmdb_id_map_ubcf,
        n_recommendations=max_k,
        user_watched_movie_ids=known_movie_ids, # Pass known_movie_ids for this user
        N_SIMILAR_USERS_PARAM=N_SIMILAR_USERS,
        user_avg_ratings_map_for_prediction=user_avg_ratings_eval_map_ubcf, # Global avg ratings for neighbors
        reverse_user_id_map=reverse_user_id_map_ubcf,
        normalized_movie_popularity_map=normalized_movie_popularity_ubcf
    )

    if not recommended_items:
        continue

    for k in k_values:
        all_precision_small_ubcf[k].append(precision_at_k(recommended_items, ground_truth_relevant_items, k))
        all_recall_small_ubcf[k].append(recall_at_k(recommended_items, ground_truth_relevant_items, k))
        all_ndcg_small_ubcf[k].append(ndcg_at_k(recommended_items, ground_truth_relevant_items, k))
        all_serendipity_small_ubcf[k].append(calculate_serendipity(recommended_items, ground_truth_relevant_items, normalized_movie_popularity_ubcf, k))

print(f"\nFinished evaluation. Metrics calculated for {processed_users_count_small_ubcf:,} test users for UBCF.")

# Calculate and print average metrics across all evaluated test users
results_ubcf_small = {'Model': 'UBCF'}
for k in k_values:
    avg_precision = np.mean(all_precision_small_ubcf[k]) if all_precision_small_ubcf[k] else 0
    avg_recall = np.mean(all_recall_small_ubcf[k]) if all_recall_small_ubcf[k] else 0
    avg_ndcg = np.mean(all_ndcg_small_ubcf[k]) if all_ndcg_small_ubcf[k] else 0
    avg_serendipity = np.mean(all_serendipity_small_ubcf[k]) if all_serendipity_small_ubcf[k] else 0

    print(f"\nMetrics @k={k} (UBCF on test_small.csv, per-user 80/20 split):\n  Average Precision: {avg_precision:.4f}\n  Average Recall:    {avg_recall:.4f}\n  Average NDCG:      {avg_ndcg:.4f}\n  Average Serendipity: {avg_serendipity:.4f}")

    results_ubcf_small[f'Precision_{k}'] = avg_precision
    results_ubcf_small[f'Recall_{k}'] = avg_recall
    results_ubcf_small[f'nDCG_{k}'] = avg_ndcg
    results_ubcf_small[f'Serendipity_{k}'] = avg_serendipity

# Convert results to DataFrame for consistency with baseline results
results_ubcf_small_df = pd.DataFrame([results_ubcf_small]).set_index('Model')

print("\nUBCF Recommendation system evaluation complete for small dataset!")
display(results_ubcf_small_df)
gc.collect()

Executing evaluation for test_small.csv (UBCF)

Starting evaluation for UBCF on test_small.csv (per-user 80/20 split), considering 582 eligible test users...


Evaluating UBCF on test_small.csv (per-user split):   0%|          | 0/582 [00:00<?, ?it/s]


Finished evaluation. Metrics calculated for 500 test users for UBCF.

Metrics @k=5 (UBCF on test_small.csv, per-user 80/20 split):
  Average Precision: 0.2368
  Average Recall:    0.0552
  Average NDCG:      0.2504
  Average Serendipity: 0.0451

Metrics @k=10 (UBCF on test_small.csv, per-user 80/20 split):
  Average Precision: 0.2014
  Average Recall:    0.0848
  Average NDCG:      0.2301
  Average Serendipity: 0.0397

Metrics @k=15 (UBCF on test_small.csv, per-user 80/20 split):
  Average Precision: 0.1816
  Average Recall:    0.1096
  Average NDCG:      0.2209
  Average Serendipity: 0.0364

UBCF Recommendation system evaluation complete for small dataset!


,Precision_5,Recall_5,nDCG_5,Serendipity_5,Precision_10,Recall_10,nDCG_10,Serendipity_10,Precision_15,Recall_15,nDCG_15,Serendipity_15
Model,,,,,,,,,,,,
UBCF,0.2368,0.055232,0.250368,0.04511,0.2014,0.084779,0.230086,0.039679,0.1816,0.10962,0.22092,0.036435


216

In [ ]:
print('Executing evaluation for test_big.csv (UBCF)')
k_values = [5, 10, 15]
all_precision_big_ubcf = {k: [] for k in k_values}
all_recall_big_ubcf = {k: [] for k in k_values}
all_ndcg_big_ubcf = {k: [] for k in k_values}
all_serendipity_big_ubcf = {k: [] for k in k_values}

# 1. Identify users for evaluation from the GLOBAL test set
# These are users who have at least MIN_RATINGS_IN_GLOBAL_TEST ratings in the 'test_big.csv' file
test_user_counts_big_ubcf = test_ratings_big['userId'].value_counts()
eligible_users_big_ubcf = test_user_counts_big_ubcf[test_user_counts_big_ubcf >= MIN_RATINGS_IN_GLOBAL_TEST].index.tolist()

print(f"\nStarting evaluation for UBCF on test_big.csv (per-user 80/20 split), considering {len(eligible_users_big_ubcf):,} eligible test users...")

processed_users_count_big_ubcf = 0
for user_id in tqdm(eligible_users_big_ubcf, desc="Evaluating UBCF on test_big.csv (per-user split)"):
    # Get ALL ratings for this specific user from the FULL 'ratings' dataset
    user_all_history = ratings[ratings['userId'] == user_id].sort_values(by='timestamp').reset_index(drop=True)

    # Perform 80/20 chronological split for THIS USER'S ENTIRE HISTORY
    user_split_point = int(len(user_all_history) * 0.8)
    user_train_individual = user_all_history.iloc[:user_split_point]
    user_test_individual = user_all_history.iloc[user_split_point:]

    # Ensure individual train set is not empty and has enough ratings for model training
    if user_train_individual.empty or len(user_train_individual) < MIN_TRAIN_RATINGS_FOR_EVAL_USER:
        continue

    # Known movies for this user (from their individual 80% train set)
    known_movie_ids = set(user_train_individual['tmdbId'])
    user_avg_rating = user_train_individual['rating'].mean()

    # Define 'relevant' items for ground truth from the INDIVIDUAL user's 20% test set
    # Rating >= 4 AND Rating >= user's average rating from their train set
    ground_truth_relevant_items = user_test_individual[
        (user_test_individual['rating'] >= 4) &
        (user_test_individual['rating'] >= user_avg_rating)
    ]['tmdbId'].tolist()

    # Only evaluate user if their individual test set has at least MIN_PREDICT_ITEMS_FOR_TEST_USER relevant items
    if not ground_truth_relevant_items or len(ground_truth_relevant_items) < MIN_PREDICT_ITEMS_FOR_TEST_USER:
        continue

    processed_users_count_big_ubcf += 1

    max_k = max(k_values)
    recommended_items = get_user_recommendations_optimized_enhanced(
        user_id,
        train_user_item_matrix_ubcf, # Use the global UBCF matrix
        nn_model,
        user_id_map_ubcf,
        tmdb_id_map_ubcf,
        n_recommendations=max_k,
        user_watched_movie_ids=known_movie_ids, # Pass known_movie_ids for this user
        N_SIMILAR_USERS_PARAM=N_SIMILAR_USERS,
        user_avg_ratings_map_for_prediction=user_avg_ratings_eval_map_ubcf, # Global avg ratings for neighbors
        reverse_user_id_map=reverse_user_id_map_ubcf,
        normalized_movie_popularity_map=normalized_movie_popularity_ubcf
    )

    if not recommended_items:
        continue

    for k in k_values:
        all_precision_big_ubcf[k].append(precision_at_k(recommended_items, ground_truth_relevant_items, k))
        all_recall_big_ubcf[k].append(recall_at_k(recommended_items, ground_truth_relevant_items, k))
        all_ndcg_big_ubcf[k].append(ndcg_at_k(recommended_items, ground_truth_relevant_items, k))
        all_serendipity_big_ubcf[k].append(calculate_serendipity(recommended_items, ground_truth_relevant_items, normalized_movie_popularity_ubcf, k))

print(f"\nFinished evaluation. Metrics calculated for {processed_users_count_big_ubcf:,} test users for UBCF.")

# Calculate and print average metrics across all evaluated test users
results_ubcf_big = {'Model': 'UBCF'}
for k in k_values:
    avg_precision = np.mean(all_precision_big_ubcf[k]) if all_precision_big_ubcf[k] else 0
    avg_recall = np.mean(all_recall_big_ubcf[k]) if all_recall_big_ubcf[k] else 0
    avg_ndcg = np.mean(all_ndcg_big_ubcf[k]) if all_ndcg_big_ubcf[k] else 0
    avg_serendipity = np.mean(all_serendipity_big_ubcf[k]) if all_serendipity_big_ubcf[k] else 0

    print(f"\nMetrics @k={k} (UBCF on test_big.csv, per-user 80/20 split):\n  Average Precision: {avg_precision:.4f}\n  Average Recall:    {avg_recall:.4f}\n  Average NDCG:      {avg_ndcg:.4f}\n  Average Serendipity: {avg_serendipity:.4f}")

    results_ubcf_big[f'Precision_{k}'] = avg_precision
    results_ubcf_big[f'Recall_{k}'] = avg_recall
    results_ubcf_big[f'nDCG_{k}'] = avg_ndcg
    results_ubcf_big[f'Serendipity_{k}'] = avg_serendipity

# Convert results to DataFrame for consistency with baseline results
results_ubcf_big_df = pd.DataFrame([results_ubcf_big]).set_index('Model')

print("\nUBCF Recommendation system evaluation complete for big dataset!")
display(results_ubcf_big_df)
gc.collect()

Executing evaluation for test_big.csv (UBCF)

Starting evaluation for UBCF on test_big.csv (per-user 80/20 split), considering 6,151 eligible test users...


Evaluating UBCF on test_big.csv (per-user split):   0%|          | 0/6151 [00:00<?, ?it/s]


Finished evaluation. Metrics calculated for 5,009 test users for UBCF.

Metrics @k=5 (UBCF on test_big.csv, per-user 80/20 split):
  Average Precision: 0.2270
  Average Recall:    0.0806
  Average NDCG:      0.2480
  Average Serendipity: 0.0431

Metrics @k=10 (UBCF on test_big.csv, per-user 80/20 split):
  Average Precision: 0.1908
  Average Recall:    0.1216
  Average NDCG:      0.2348
  Average Serendipity: 0.0374

Metrics @k=15 (UBCF on test_big.csv, per-user 80/20 split):
  Average Precision: 0.1711
  Average Recall:    0.1540
  Average NDCG:      0.2328
  Average Serendipity: 0.0342

UBCF Recommendation system evaluation complete for big dataset!


,Precision_5,Recall_5,nDCG_5,Serendipity_5,Precision_10,Recall_10,nDCG_10,Serendipity_10,Precision_15,Recall_15,nDCG_15,Serendipity_15
Model,,,,,,,,,,,,
UBCF,0.226991,0.080557,0.247961,0.043145,0.190777,0.12157,0.234842,0.037425,0.171052,0.154006,0.232756,0.034213


1152